In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, epoch):
    model.train()
    total_loss = 0
    
    for batch_idx, (data, _) in enumerate(loader):
        data = data.to(device)
        
        optimizer.zero_grad()
        
        # Mixed Precision Context
        with autocast():
            # Forward pass
            # model inputs: (B, N, 4)
            # outputs: pred (B, N, 4), cls_logits (B, C), mask (B, N)
            pred, cls_logits, mask = model(data)
            
            # MAE Loss on masked patches only
            # Verify: loss = sum(||p_target - p_pred||^2 * mask) / sum(mask)
            loss = mae_loss(pred, data, mask)
        
        # Scale loss and backprop
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch} | Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}")
            
        # Checkpoint every 500 batches (or fewer for test)
        if batch_idx > 0 and batch_idx % 500 == 0:
             save_checkpoint(epoch, model, optimizer, scaler, CHECKPOINT_PATH)
             
    return total_loss / len(loader)

# --- SMOKE TEST ---
print("Running Smoke Test...")
smoke_loader = DataLoader(dataset=train_dataset, batch_size=2, shuffle=True)
# Run just 1 epoch on limited data usually, but here we just iterate a few steps
model.train()
try:
    data, _ = next(iter(smoke_loader))
    data = data.to(device)
    with autocast():
        pred, _, mask = model(data)
        loss = mae_loss(pred, data, mask)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    print("Smoke Test Passed: Forward/Backward pass successful. No OOM.")
except Exception as e:
    print(f"Smoke Test Failed: {e}")
    raise e

# --- FULL TRAINING START ---
start_epoch = load_checkpoint(CHECKPOINT_PATH, model, optimizer, scaler)
NUM_EPOCHS = 1 # Set to higher for actual training

for epoch in range(start_epoch, NUM_EPOCHS):
    avg_loss = train_one_epoch(model, train_loader, optimizer, scaler, epoch)
    print(f"Epoch {epoch} Completed. Avg Loss: {avg_loss:.4f}")
    save_checkpoint(epoch, model, optimizer, scaler, CHECKPOINT_PATH)

print("Training script configuration complete.")


## 5. Training Loop and Smoke Test

Run the training loop. We first verify with a **Smoke Test** configuration as customary in GSoC projects to ensure pipeline stability before full-scale runs.


In [ ]:
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scaler = GradScaler()

CHECKPOINT_PATH = Path("D:/datasets/mae_checkpoint.pt")

def save_checkpoint(epoch, model, optimizer, scaler, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
    }, path)
    print(f"Checkpoint saved to {path}")

def load_checkpoint(path, model, optimizer, scaler):
    if path.exists():
        checkpoint = torch.load(path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resumed training from epoch {start_epoch}")
        return start_epoch
    return 0


## 4. Efficiency: AMP and Checkpointing

We implement the training loop with:
- **Automatic Mixed Precision (AMP)**: Uses `torch.cuda.amp.GradScaler` to speed up training and reduce memory.
- **Gradient Checkpointing**: Logic is handled within the model's `transformer_blocks` loop (controlled by `model.train()` state).
- **Checkpoint Resilience**: A saver usually saves to Drive/Disk.


In [ ]:
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader

class JetDataset(Dataset):
    def __init__(self, file_path, num_particles=128, limit=None):
        self.num_particles = num_particles
        self.file_path = Path(file_path)
        
        if not self.file_path.exists():
            print(f"Warning: {file_path} not found. using synthetic data for demonstration.")
            self.data = torch.randn(100, num_particles, 4) # (E, px, py, pz)
            self.labels = torch.randint(0, 2, (100,))
            self.length = 100
        else:
            # Simple H5 loader logic (adjust key names based on actual file structure)
            if self.file_path.suffix == '.h5':
                with h5py.File(self.file_path, 'r') as f:
                    # Assuming standard structure, needing adjustment for real JetClass keys
                    # e.g., 'particles', 'jet_class'
                    # Load a subset for efficiency
                    if limit:
                        self.data = torch.tensor(f['particles'][:limit], dtype=torch.float32)
                        self.labels = torch.tensor(f['labels'][:limit], dtype=torch.long)
                    else:
                        self.data = torch.tensor(f['particles'][:], dtype=torch.float32)
                        self.labels = torch.tensor(f['labels'][:], dtype=torch.long)
            elif self.file_path.suffix == '.npz':
                data = np.load(self.file_path)
                # Assuming 'X' and 'y' keys
                if limit:
                     self.data = torch.tensor(data['X'][:limit], dtype=torch.float32)
                     self.labels = torch.tensor(data['y'][:limit], dtype=torch.long)
                else: 
                     self.data = torch.tensor(data['X'], dtype=torch.float32)
                     self.labels = torch.tensor(data['y'], dtype=torch.long)

            # Preprocessing: Pad/Truncate
            # Assumes data comes as (N, P, 4) or similar. 
            # If (N, 4, P) transpose.
            if self.data.shape[1] == 4: 
                self.data = self.data.transpose(1, 2)
                
            curr_particles = self.data.shape[1]
            if curr_particles > num_particles:
                self.data = self.data[:, :num_particles, :]
            elif curr_particles < num_particles:
                padding = torch.zeros(self.data.shape[0], num_particles - curr_particles, 4)
                self.data = torch.cat([self.data, padding], dim=1)
                
            self.length = len(self.labels)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Create DataLoader
# Pointing to the file we (tried to) download
dataset_path = Path("D:/datasets/JetClass_Part0.h5") 
train_dataset = JetDataset(dataset_path, num_particles=128, limit=1000) # Limit for demo/smoke test
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)

print(f"Dataset size: {len(train_dataset)}")


## 3. Data Loading

We'll create a simple Dataset class to load the H5/NumPy files. For this demonstration, we'll create a synthetic dataset if real files are not immediately available or essentially just load the raw arrays.


In [ ]:
from model import HybridPhysicsMAE, mae_loss

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Instantiate the model
model = HybridPhysicsMAE(
    input_dim=4,        # (E, px, py, pz)
    embed_dim=128,      # Embedding dimension
    num_heads=8,        # Attention heads
    num_layers=4,       # Transformer layers
    num_classes=2,      # Jet tagging classes (e.g. Quark vs Gluon, or JetClass categories)
    mask_ratio=0.5      # 50% masking
).to(device)

print(model)


## 2. Model Architecture

We import the `HybridPhysicsMAE` model from our `model.py`. This model combines:
1.  **L-GATr Encoder**: For equivariant feature extraction from 4-vectors.
2.  **MAE Masking**: Randomly masks a portion of the input particles.
3.  **Transformer Backbone (ParT)**: Processes the visible particles.
4.  **Reconstruction Head**: Reconstructs the masked particles.
5.  **Classification Head**: Predicts jet class (for fine-tuning/auxiliary loss).


In [ ]:
import os
import sys
from pathlib import Path
import torch

# Ensure we can import from the current directory
sys.path.append(os.getcwd())

# Run the data download script
# This script manages the download of JetClass (Part 0) and QuarkGluon datasets
%run download_data.py

# Check if data exists
dataset_dir = Path("D:/datasets")
if dataset_dir.exists():
    print(f"Data directory confirmed at {dataset_dir}")
    print("Files:", list(dataset_dir.glob("*")))
else:
    print("Data directory not found. Please check download_data.py execution.")


# HybridPhysicsMAE Training with AMP and Checkpointing

This notebook implements the training pipeline for the Hybrid L-GATr + ParT model with a Masked Autoencoder (MAE) objective.

## 1. Environment Setup and Data Acquisition

First, we ensure the necessary data is downloaded. We will use the `download_data.py` script we created.
